# Llama-2-7B next-token log-probs for 100k prompts of one-million-instructions

This is the scaled-up data for the T5-base inverter. Llama-2-7B reads each prompt, and its
next-token log-probabilities are saved, **with the same method as the teammate's
`llama2/llama2_extraction.ipynb`**:
- the model is loaded in 8-bit (LLM.int8)
- padding goes on the left, using `</s>`
- prompts are cut at 512 tokens
- the vector is the `log_softmax` of the last position's logits, in float32

The weights come from the ungated mirror `NousResearch/Llama-2-7b-hf` (same files as
`meta-llama/Llama-2-7b-hf`).

**Steps**
1. **Select prompts** (`select_one_million_prompts.py`). The rules:
   - prompts without a system prompt only
   - no Dolly prompts, so the Dolly validation set stays unseen
   - no duplicates
   - at most 64 T5 tokens
   - a seeded random sample of `N_PROMPTS`
2. **Check the method** on the first `VERIFY_ROWS` Dolly prompts, by comparing with the
   teammate's stored vectors:
   - once with their exact batching (batches of 4, original order);
   - once with the batching used here (batches of 32, sorted by length).
3. **Generate the vectors** (`extract_llama2_logprobs.py`), one 8-bit Llama per GPU, each
   filling its share of the rows of `vectors.npy`.
4. **Check the result:** every row must be a normalized log-probability vector.

**Output:** `/kaggle/working/one_million_llama2/` holds `instructions.npy`, `sources.npy`,
`vectors.npy` (float32, N × 32000), `selection.json` and `verify.json`. Accelerator: GPU T4 ×2,
with internet on.

In [ ]:
SMOKE = True
N_PROMPTS = 2000 if SMOKE else 101_000   # ~1% becomes the validation split at training time
VERIFY_ROWS = 400
LLAMA = "NousResearch/Llama-2-7b-hf"

REPO_URL = "https://github.com/Yae-sine/Language-Model-Inversion.git"
REPO_BRANCH = "main"
REPO_DIR = "/tmp/lmi"
V2T_DIR = f"{REPO_DIR}/vec2text"
NPY_DIR = "/tmp/llama2_dolly15k"          # teammate's Dolly vectors (from the dataset input)
DATASET_DIR = "/tmp/dolly_llama2_inv"     # Dolly HF dataset (same split as before)
WORK = "/kaggle/working"

OUT = f"{WORK}/one_million_llama2"

In [ ]:
import glob, json, os, shutil, subprocess, sys, zipfile


def sh(cmd, cwd=None, env=None, log=None):
    # Run a shell command, stream its output into the notebook, fail loudly on error.
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env=env, executable="/bin/bash",
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    f = open(log, "a") if log else None
    for line in p.stdout:
        print(line, end="", flush=True)
        if f:
            f.write(line)
            f.flush()
    rc = p.wait()
    if f:
        f.close()
    if rc != 0:
        raise RuntimeError(f"command failed with exit code {rc}: {cmd}")


sh("nvidia-smi")
sh(f"rm -rf {REPO_DIR} && git clone --depth 1 -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}")
sh(f"git -C {REPO_DIR} log --oneline -1")

## Environment

`transformers` is pinned to 4.45.2. That is the last release that still accepts the
`evaluation_strategy` setting vec2text's `TrainingArguments` uses (removed in 4.46). It is also the
first release that works with `tokenizers` 0.20, which ships Python 3.13 builds; older `tokenizers`
fail to install on Kaggle's Python 3.13. A constraints file keeps Kaggle's own torch build.

`torchvision` is uninstalled. When it is loaded, `datasets` 3.x's torch formatter imports
`torchvision.io.VideoReader`, which recent torchvision removed, so reading the dataset crashes.
vec2text never uses torchvision.

In [ ]:
torch_ver = subprocess.check_output(
    [sys.executable, "-c", "import torch; print(torch.__version__)"], text=True).strip()
with open("/tmp/constraints.txt", "w") as f:
    f.write(f"torch=={torch_ver}\n")
print("keeping torch", torch_ver)

sh('pip install -q -c /tmp/constraints.txt "transformers==4.45.2" "tokenizers==0.20.3" "datasets>=2.20,<4" '
   '"accelerate==0.34.2" "sentence-transformers==3.0.1" evaluate sacrebleu rouge_score nltk '
   'sentencepiece einops tenacity openai')
sh("pip uninstall -y -q torchvision")
sh(f"pip install -q -e {V2T_DIR} --no-deps")
sh("python -c \"import nltk; nltk.download('punkt', quiet=True); nltk.download('punkt_tab', quiet=True)\"")
sh("python -c \"import torch, transformers, datasets, vec2text; print('torch', torch.__version__, '| transformers', transformers.__version__, '| datasets', datasets.__version__, '| cuda', torch.cuda.is_available())\"")

## Inputs

The files are found by searching `/kaggle/input`, so this works whatever the dataset's mount path is,
and whether or not Kaggle unzipped the `.npz`.

The Llama-2 tokenizer and config are copied to `/kaggle/working/meta-llama/Llama-2-7b-hf`. Every
command then runs from `/kaggle/working`, so the model name `meta-llama/Llama-2-7b-hf` resolves to
that local folder, and the gated Hugging Face repo is never contacted.

In [ ]:
def find(name):
    return sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))


os.makedirs(NPY_DIR, exist_ok=True)
npz = [p for p in find("llama2_dolly15k.npz") if os.path.isfile(p) and zipfile.is_zipfile(p)]
if npz:
    print("extracting", npz[0])
    with zipfile.ZipFile(npz[0]) as z:
        for member in ("instructions.npy", "vectors.npy"):
            if not os.path.exists(f"{NPY_DIR}/{member}"):
                z.extract(member, NPY_DIR)
else:  # Kaggle already unzipped it
    for member in ("instructions.npy", "vectors.npy"):
        src = find(member)[0]
        print("linking", src)
        if not os.path.exists(f"{NPY_DIR}/{member}"):
            os.symlink(src, f"{NPY_DIR}/{member}")

tok_dir = os.path.dirname(find("tokenizer.model")[0])
llama_dir = f"{WORK}/meta-llama/Llama-2-7b-hf"
os.makedirs(llama_dir, exist_ok=True)
for name in ("config.json", "generation_config.json", "special_tokens_map.json",
             "tokenizer_config.json", "tokenizer.json", "tokenizer.model"):
    shutil.copy(f"{tok_dir}/{name}", llama_dir)

sh(f"ls -la {NPY_DIR} {llama_dir}")

## Llama-2-7B and bitsandbytes

The weights (13.5 GB, safetensors only) are downloaded once, before the per-GPU workers
start, so the workers don't download them at the same time.

In [ ]:
sh('pip install -q -c /tmp/constraints.txt "bitsandbytes==0.50.2"')
sh("python -c \"import bitsandbytes, torch; print('bitsandbytes', bitsandbytes.__version__, '| GPUs', torch.cuda.device_count())\"")
sh(f"python -c \"import huggingface_hub as h; print(h.snapshot_download('{LLAMA}', "
   "allow_patterns=['*.json', '*.safetensors', 'tokenizer.model']))\"")
sh("df -h / /tmp /kaggle/working | sort -u; free -g")

## 1. Select the prompts

In [ ]:
if not os.path.exists(f"{OUT}/instructions.npy"):
    sh(f"python {V2T_DIR}/experiment_scripts/logits/select_one_million_prompts.py "
       f"--out_dir {OUT} --n {N_PROMPTS} --exclude_instructions {NPY_DIR}/instructions.npy")
print(json.dumps(json.load(open(f"{OUT}/selection.json")), indent=1)[:4000])

## 2. Check the method against the teammate's Dolly vectors

These Dolly prompts are re-encoded with this pipeline and compared with the teammate's
vectors, using:
- the KL divergence between the two distributions,
- how often the most likely token is the same,
- the largest log-prob difference over the teammate's top-20 tokens.

In [ ]:
import numpy as np

def run_extract(instr, vec, extra, gpu=0, log=None):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1")
    sh(f"python {V2T_DIR}/experiment_scripts/logits/extract_llama2_logprobs.py "
       f"--model {LLAMA} --instructions {instr} --vectors {vec} {extra}", env=env, log=log)

def compare(ref, new):
    ref = ref.astype(np.float64); new = new.astype(np.float64)
    p = np.exp(ref)
    kl = (p * (ref - new)).sum(1)
    top1 = (ref.argmax(1) == new.argmax(1)).mean()
    top20 = np.argsort(-ref, axis=1)[:, :20]
    d20 = np.abs(np.take_along_axis(ref, top20, 1) - np.take_along_axis(new, top20, 1)).max(1)
    lse = np.log(np.exp(new).sum(1))
    return {"rows": len(ref), "kl_mean": float(kl.mean()), "kl_max": float(kl.max()),
            "top1_agreement": float(top1), "top20_max_abs_diff_mean": float(d20.mean()),
            "top20_max_abs_diff_max": float(d20.max()), "logsumexp_max_abs": float(np.abs(lse).max())}

vdir = "/tmp/verify"
os.makedirs(vdir, exist_ok=True)
ref_instr = np.load(f"{NPY_DIR}/instructions.npy", mmap_mode="r")
ref_vec = np.load(f"{NPY_DIR}/vectors.npy", mmap_mode="r")
np.save(f"{vdir}/instructions.npy", np.array(ref_instr[:VERIFY_ROWS]))
verify = {}
for name, extra in [("teammate_batching_bs4", "--batch_size 4 --sort_by_length 0"),
                    ("pipeline_batching_bs32_sorted", "--batch_size 32 --sort_by_length 1")]:
    vec = f"{vdir}/vectors_{name}.npy"
    np.lib.format.open_memmap(vec, mode="w+", dtype=np.float32, shape=(VERIFY_ROWS, 32000)).flush()
    run_extract(f"{vdir}/instructions.npy", vec, extra + " --log_every 25")
    verify[name] = compare(np.asarray(ref_vec[:VERIFY_ROWS]), np.load(vec))
    print(name, json.dumps(verify[name], indent=1), flush=True)
json.dump(verify, open(f"{OUT}/verify.json", "w"), indent=1)

## 3. Generate the vectors on every GPU

One worker per GPU, each with its own 8-bit Llama, fills rows `i % n_gpus == k` of the same
`vectors.npy`. Progress is printed every couple of minutes.

In [ ]:
import time
import torch

n_gpus = torch.cuda.device_count()
procs = []
for g in range(n_gpus):
    log = f"{WORK}/extract_gpu{g}.log"
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g), PYTHONUNBUFFERED="1")
    cmd = (f"python {V2T_DIR}/experiment_scripts/logits/extract_llama2_logprobs.py "
           f"--model {LLAMA} --instructions {OUT}/instructions.npy --vectors {OUT}/vectors.npy "
           f"--shard {g}/{n_gpus} --batch_size 32 --sort_by_length 1 --log_every 100")
    print("$", cmd, flush=True)
    procs.append(subprocess.Popen(cmd, shell=True, env=env, stdout=open(log, "w"),
                                  stderr=subprocess.STDOUT))
t0 = time.time()
while any(p.poll() is None for p in procs):
    time.sleep(120)
    for g in range(n_gpus):
        lines = [l for l in open(f"{WORK}/extract_gpu{g}.log").read().splitlines() if l.strip()]
        print(f"[{(time.time() - t0) / 60:5.1f} min] gpu{g}: {lines[-1][:200] if lines else ''}", flush=True)
for g, p in enumerate(procs):
    if p.returncode != 0:
        print(open(f"{WORK}/extract_gpu{g}.log").read()[-5000:])
        raise RuntimeError(f"worker on gpu{g} failed with exit code {p.returncode}")
print(f"all workers done in {(time.time() - t0) / 60:.1f} min")

## 4. Check the result

In [ ]:
import transformers

vec = np.load(f"{OUT}/vectors.npy", mmap_mode="r")
instr = np.load(f"{OUT}/instructions.npy", mmap_mode="r")
bad, top1 = [], []
for i in range(0, len(vec), 2000):
    v = np.asarray(vec[i : i + 2000], dtype=np.float64)
    m = v.max(1, keepdims=True)
    lse = (m[:, 0] + np.log(np.exp(v - m).sum(1)))
    ok = np.isfinite(v).all(1) & (np.abs(lse) < 1e-3)
    bad += [i + j for j in np.where(~ok)[0]]
    top1 += v.argmax(1).tolist()
print(f"{len(vec)} vectors, {len(bad)} invalid (not finite or not normalized)")
assert not bad, f"invalid rows, e.g. {bad[:10]}"
tok = transformers.AutoTokenizer.from_pretrained(LLAMA)
counts = np.bincount(top1, minlength=32000)
print("most frequent top-1 tokens:",
      [(repr(tok.decode([t])), int(counts[t])) for t in np.argsort(-counts)[:10]])
for i in range(3):
    print(repr(str(instr[i])[:100]), "->", [repr(tok.decode([t])) for t in np.argsort(-vec[i])[:5]])
sh("df -h / /tmp /kaggle/working | sort -u; free -g")